# 04. Build dataset B

Dataset B = dataset A (the 640 queries, unchanged) plus about 860 new, harder queries.
The goal is a dataset where classifiers do not score near 99%.

How the new rows are made: template families written with the help of Claude Code, expanded
with a fixed seed (42). The label of each query follows the written rules in `label_rules.md`
(each family is tied to one tool by those rules). No language model labels individual queries,
and the labels have not been verified by hand. Quality is checked with automated tests below.

Difficulty is set by design (new families), not by tuning against any model's score.

In [1]:
import random, re
from collections import Counter
import pandas as pd

SEED = 42
rng = random.Random(SEED)

A = pd.read_csv("cyber_logs.csv")
print("Dataset A:", len(A), "rows")

AGENT_OF = {
    "SSHConnect": "attacker", "NmapScan": "attacker", "PortScan": "attacker",
    "CheckVulnerability": "attacker",
    "ReadAuthLog": "defender", "ListeningPorts": "defender", "BlockIP": "defender",
    "CheckFailedLogins": "defender", "ListProcesses": "defender",
    "GetSystemInfo": "shared", "ReadSyslog": "shared",
}
TOOLS = sorted(AGENT_OF)

Dataset A: 640 rows


## 1. Target size per tool

Total 1500. Each tool is topped up toward the same count (136, plus one extra for the four
tools that have the fewest rows in A so the total is exactly 1500).

In [2]:
a_counts = A["tool_ground_truth"].value_counts().to_dict()
target = {t: 136 for t in TOOLS}
for t in sorted(TOOLS, key=lambda t: (a_counts[t], t))[:4]:
    target[t] += 1
need = {t: target[t] - a_counts[t] for t in TOOLS}
assert sum(target.values()) == 1500
print("new rows needed per tool:", need, "| total new:", sum(need.values()))

new rows needed per tool: {'BlockIP': 81, 'CheckFailedLogins': 88, 'CheckVulnerability': 55, 'GetSystemInfo': 88, 'ListProcesses': 81, 'ListeningPorts': 76, 'NmapScan': 69, 'PortScan': 68, 'ReadAuthLog': 83, 'ReadSyslog': 79, 'SSHConnect': 92} | total new: 860


## 2. Template families (new rows)

Every new row is tagged with a `category` (the family type) and a `family` id (the specific
template), so evaluation can hold out whole templates.

- `shared_vocab`: the query uses words that belong to a neighbouring tool, and the rule decides.
- `no_cue`: states a goal and avoids naming the tool.
- `near_miss`: wording that leans toward a confusable neighbouring tool.
- `vocab_shift`: synonyms and verbs not used in dataset A.
- `cross_role`: worded from the other side's point of view.
- `messy`: casual text with typos, abbreviations and filler words, built from the other templates.

In [3]:
T = ["192.168.56.102", "the target", "the defender", "the target machine",
     "the web server", "that box", "10.0.0.114"]
H = ["", " on this host", " on this box", " on this machine", " here", " on the server"]
W = ["", " today", " since boot", " overnight", " in the last hour", " this morning"]
IPS = ["10.0.0.188", "the attacker address", "that suspicious IP", "the noisy host",
       "203.0.113.7", "the flagged address"]
SVC = ["ssh", "OpenSSH", "Apache", "nginx", "MySQL", "vsftpd", "the web server", "samba"]
VER = ["7.2", "8.2", "2.4.49", "1.18", "8.0.32", "2.3.4"]

FAM = {}   # (category, tool) -> list of templates

def add(cat, tool, templates):
    FAM[(cat, tool)] = templates

# ---- shared_vocab ----
add("shared_vocab", "NmapScan", [
    "Beyond just open ports, tell me which services and versions {t} runs",
    "I want service names and versions from {t}, not only a port list",
    "Run version detection on {t} so I know what software answers",
    "Map {t} to the software versions behind each open port",
    "Which versions of the services on {t} are exposed{w}?"])
add("shared_vocab", "PortScan", [
    "Just the open ports on {t}, skip the service versions",
    "List which ports respond on {t}, I do not need version details",
    "Only tell me the port numbers that are open on {t}",
    "Sweep {t} for open ports and leave out service detection",
    "Which ports are open on {t}? Port numbers only"])
add("shared_vocab", "CheckVulnerability", [
    "Do not just list services, tell me if {s} {v} on {t} is exploitable",
    "Is {s} {v} known to be vulnerable, or just present?",
    "Beyond the version, does {s} {v} have published CVEs?",
    "Check whether {s} {v} has a known exploit, not only what it is",
    "Is the {s} on {t} actually exploitable{w}?"])
add("shared_vocab", "SSHConnect", [
    "Do not scan {t}, just open an SSH session to it",
    "Skip the recon and get a shell on {t} over SSH",
    "Connect to {t} with SSH instead of probing it",
    "Rather than scanning {t}, log in to it remotely",
    "Open a remote shell on {t}, no scanning needed"])
add("shared_vocab", "ReadAuthLog", [
    "Show the whole auth log{w}, not only the failed logins",
    "I want every line of the authentication log{h}, successes included",
    "Read the full auth log{h}, not a filtered view",
    "Pull the complete authentication log{h}{w}",
    "All login events{h}{w}, successful and failed"])
add("shared_vocab", "CheckFailedLogins", [
    "Only the failed password attempts{h}, not the full auth log",
    "From the auth log, just the failed logins{w}",
    "Filter the authentication log down to failures{h}",
    "Count the bad login attempts{h}{w}, skip the successful ones",
    "Show failed SSH logins only{h}"])
add("shared_vocab", "ReadSyslog", [
    "Read the general system log{h}, not the auth log",
    "I want syslog{h}, not the authentication log",
    "Show recent system messages{h}{w}, nothing from auth",
    "Tail the system log{h}, the main one, not auth.log",
    "Show syslog entries{h}{w}"])
add("shared_vocab", "ListeningPorts", [
    "Which sockets are listening{h}? I do not mean processes",
    "Show listening ports{h}, not the process list",
    "List what is bound to a port{h}{w}",
    "Which services are accepting connections{h}, by port?",
    "Show the listening sockets{h}, not the running programs"])
add("shared_vocab", "ListProcesses", [
    "Which programs are running{h}{w}? Not the open ports",
    "Show the busiest processes{h}, not the listening sockets",
    "Top CPU users{h}, not network ports",
    "List running processes{h}{w}, skip the network side",
    "What is using the CPU{h}{w}?"])
add("shared_vocab", "GetSystemInfo", [
    "Hostname, kernel and uptime{h}, no logs please",
    "Just the OS version and uptime{h}, skip the logs",
    "Which kernel is this{h}, and how long has it been up?",
    "Give me system details{h}, not the log files",
    "Uptime and kernel version{h}{w}"])
add("shared_vocab", "BlockIP", [
    "Do not just log {ip}, block it",
    "Actually block {ip} at the firewall, reading logs is not enough",
    "Stop {ip} from connecting, do not only watch it",
    "Deny {ip} at the firewall instead of investigating further",
    "Drop traffic from {ip} now"])

# ---- no_cue ----
add("no_cue", "NmapScan", [
    "What is running on {t}?", "What software does {t} expose{w}?",
    "Figure out what {t} is hosting",
    "I need to know what kind of services {t} offers",
    "Find out which applications answer on {t}"])
add("no_cue", "PortScan", [
    "Which doors are open on {t}?", "Is anything reachable on {t}?",
    "What can I reach on {t} from here?", "See which entry points {t} has",
    "Which ports does {t} answer on?"])
add("no_cue", "CheckVulnerability", [
    "Could the {s} on {t} be abused?", "Is {s} {v} a security risk?",
    "Is there a way in through {s} {v}?",
    "Would {s} {v} be safe to leave exposed?", "How risky is the {s} on {t}?"])
add("no_cue", "SSHConnect", [
    "Get me a shell on {t}", "I need to get onto {t} remotely",
    "Put me on {t} so I can run commands", "Give me remote access to {t}",
    "I want a session on {t}"])
add("no_cue", "ReadAuthLog", [
    "Who logged in{h}{w}?", "Show me the login history{h}",
    "Who has accessed this machine lately?", "Which accounts signed in{h}{w}?",
    "When did people sign in{h}?"])
add("no_cue", "CheckFailedLogins", [
    "Is anyone guessing passwords{h}?", "Are there signs of brute force{h}{w}?",
    "Is someone trying to break in through SSH{h}?",
    "Did anyone get their password wrong a lot{h}{w}?",
    "Has anyone been hammering the login{h}?"])
add("no_cue", "ReadSyslog", [
    "Did anything odd get logged{h}{w}?", "What has the system been reporting{h}?",
    "Anything unusual in the machine logs{h}{w}?",
    "What messages has the OS been writing{h}?", "Check the system messages{h}{w}"])
add("no_cue", "ListeningPorts", [
    "What is this machine exposing to the network{w}?",
    "Which doors are open on this host from the inside?",
    "What can others reach on this box{w}?",
    "What does this host offer to the network?",
    "What is this machine listening for?"])
add("no_cue", "ListProcesses", [
    "Why is this machine so slow{w}?", "What is eating the CPU{h}?",
    "What is hogging resources{h}{w}?", "What is this machine busy doing{w}?",
    "What is running{h} right now?"])
add("no_cue", "GetSystemInfo", [
    "What am I running on{h}?", "How long has this machine been up?",
    "What kind of box is this?", "Tell me about this system{w}", "What OS is this?"])
add("no_cue", "BlockIP", [
    "Keep {ip} away from this machine", "Cut {ip} off from this host",
    "{ip} should not reach us anymore", "Make {ip} go away",
    "I do not want {ip} talking to this box"])

# ---- near_miss ----
add("near_miss", "NmapScan", [
    "Scan {t} and tell me what is there", "Scan {t}, ports and what is behind them",
    "Check {t} in depth for services behind each port",
    "Probe {t} and identify what answers on each port",
    "Do a thorough scan of {t} with service identification"])
add("near_miss", "PortScan", [
    "Run nmap against {t} for the open ports", "Use nmap to see which ports are open on {t}",
    "nmap {t} and give me only open ports", "Nmap sweep of {t}: open ports please",
    "Quick nmap of {t}, which ports are open?"])
add("near_miss", "CheckVulnerability", [
    "Scan {t} for weaknesses in {s}", "Test {s} {v} on {t} for known flaws",
    "Check {t} for vulnerable {s}", "Look for exploitable software on {t}",
    "Scan {t} and report any known CVEs"])
add("near_miss", "SSHConnect", [
    "Reach {t} on port 22 and log in", "Connect to port 22 on {t}",
    "Talk to the SSH service on {t} and open a session",
    "Use port 22 to get a shell on {t}", "Authenticate to the SSH server on {t}"])
add("near_miss", "ReadAuthLog", [
    "Check the login attempts{h}", "Look at who tried to log in{h}{w}",
    "Review login activity{h}{w}", "Look through auth.log{h} for anything about logins",
    "Show the authentication events{h}"])
add("near_miss", "CheckFailedLogins", [
    "Check the auth log for password failures{h}", "Search auth.log{h} for rejected logins",
    "Look in auth.log for wrong passwords{h}{w}",
    "Which login attempts in auth.log failed{h}?",
    "Find rejected SSH attempts in the auth log{h}"])
add("near_miss", "ReadSyslog", [
    "Look at the logs{h}{w}", "Check /var/log{h} for messages", "Read the machine logs{h}",
    "Show log messages{h}{w}", "Check the log files{h}"])
add("near_miss", "ListeningPorts", [
    "Which services are running and on which ports{h}?",
    "Show what is running on the network side{h}", "What is bound to which port{h}?",
    "List the services that listen{h}", "Which programs hold open ports{h}?"])
add("near_miss", "ListProcesses", [
    "Which services are running{h}{w}?", "List the running services{h}",
    "What programs are up{h}{w}?", "Show everything that is running{h}",
    "What is currently active{h}{w}?"])
add("near_miss", "GetSystemInfo", [
    "Give me an overview of this machine{w}", "Summarize the state of this host",
    "Quick status check{h}", "Check the health of this box{w}",
    "Give me the basics about this host"])
add("near_miss", "BlockIP", [
    "The failed logins came from {ip}, shut it out", "{ip} keeps failing to log in, ban it",
    "Since {ip} is attacking, stop its traffic", "{ip} is the source of the probes, block it",
    "Block the address behind the failed logins: {ip}"])

# ---- vocab_shift ----
add("vocab_shift", "NmapScan", [
    "Enumerate the services on {t}", "Fingerprint {t} and list the software versions",
    "Footprint {t}: which service versions run there?",
    "Interrogate {t} for service banners", "Catalogue the applications running on {t}"])
add("vocab_shift", "PortScan", [
    "Sweep {t} for listening ports", "Probe {t} for reachable ports",
    "Enumerate the open ports on {t}", "Discover the open ports on {t}",
    "Map the exposed ports of {t}"])
add("vocab_shift", "CheckVulnerability", [
    "Audit {s} {v} for known flaws", "Look up advisories for {s} {v}",
    "Research known weaknesses in {s} {v}", "Cross-check {s} {v} against known exploits",
    "Assess the exposure of {s} {v}"])
add("vocab_shift", "SSHConnect", [
    "Establish a remote shell to {t}", "Initiate an SSH session with {t}",
    "Tunnel in to {t} with SSH", "Remote in to {t} using SSH",
    "Open a terminal on {t} over SSH"])
add("vocab_shift", "ReadAuthLog", [
    "Pull up the authentication records{h}", "Dump the auth trail{h}{w}",
    "Inspect the login records{h}", "Review the access log for sign-ins{h}",
    "Print the auth journal{h}"])
add("vocab_shift", "CheckFailedLogins", [
    "Audit the rejected sign-ins{h}{w}", "Tally the denied logins{h}",
    "Review unsuccessful authentication attempts{h}", "Enumerate credential failures{h}{w}",
    "List the refused sign-in attempts{h}"])
add("vocab_shift", "ReadSyslog", [
    "Inspect the system journal{h}", "Dump the OS event log{h}{w}",
    "Pull up the kernel and service messages{h}",
    "Review the machine event messages{h}", "Print the general system log{h}"])
add("vocab_shift", "ListeningPorts", [
    "Enumerate the bound sockets{h}", "Inspect the open server sockets{h}",
    "Audit the network listeners{h}", "Dump the listening endpoints{h}",
    "Review which ports the host serves on"])
add("vocab_shift", "ListProcesses", [
    "Inspect the running tasks{h}{w}", "Dump the process table{h}",
    "Audit the active processes{h}", "Review what is executing{h}{w}",
    "Print the task list sorted by CPU{h}"])
add("vocab_shift", "GetSystemInfo", [
    "Report the platform details{h}", "Print the OS release and uptime{h}",
    "Identify the host and its kernel{h}", "Dump basic machine facts{h}",
    "Show platform and boot time{h}"])
add("vocab_shift", "BlockIP", [
    "Blacklist {ip} at the perimeter", "Null-route {ip}", "Shun {ip} on the firewall",
    "Reject all packets from {ip}", "Ban {ip} from reaching this host"])

# ---- cross_role ----
add("cross_role", "ListeningPorts", [
    "Show me what an outsider could connect to on this host",
    "What would a scanner find open here?",
    "List the exposure an attacker would see on this box"])
add("cross_role", "ReadAuthLog", [
    "Would my logins show up on this host? Show the auth log",
    "See what a login leaves behind in the auth log{h}",
    "Show the record an intruder login would create{h}"])
add("cross_role", "CheckFailedLogins", [
    "Would an attacker's wrong passwords show up? Show failed logins{h}",
    "Show the failed attempts a brute force would leave{h}"])
add("cross_role", "PortScan", [
    "Find the open ports a defender might forget to close on {t}",
    "List the open ports on {t} that an admin would watch"])
add("cross_role", "NmapScan", [
    "Find the services on {t} that a defender would be patching",
    "List the service versions on {t} an admin must keep current"])
add("cross_role", "CheckVulnerability", [
    "Would a defender worry about {s} {v}? Check its known flaws",
    "Check whether {s} {v} is on an admin patch list for known exploits"])
add("cross_role", "SSHConnect", [
    "Log in to {t} the way an admin would, over SSH",
    "Open the same SSH session an admin uses on {t}"])
add("cross_role", "ReadSyslog", [
    "Show the system messages an admin would watch{h}",
    "Show what the OS logs that an intruder might trigger{h}"])
add("cross_role", "ListProcesses", [
    "Show the processes an intruder might have started{h}",
    "List what is running that a defender should review{h}"])
add("cross_role", "GetSystemInfo", [
    "Give the host facts an attacker would gather first{h}",
    "Show the system details a defender keeps on record{h}"])
add("cross_role", "BlockIP", [
    "Do what a defender would do about {ip}: block it",
    "Shut {ip} out the way an admin would"])

print("template pools:", len(FAM), "| templates:", sum(len(v) for v in FAM.values()))

template pools: 55 | templates: 244


## 3. Expand the templates

Round-robin over the families for each tool, with fixed-seed filler choices. `messy` rows take
a base template (from shared_vocab, no_cue or vocab_shift) and add casual noise; they keep the
base template's family id, so a whole template can be held out together.

In [4]:
def fill(tmpl):
    s = tmpl.format(t=rng.choice(T), h=rng.choice(H), w=rng.choice(W),
                    ip=rng.choice(IPS), s=rng.choice(SVC), v=rng.choice(VER))
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+([?,.:])", r"\1", s)
    return s

ABBR = {"authentication": "auth", "information": "info", "processes": "procs",
        "address": "addr", "connection": "conn", "versions": "vers",
        "machine": "box", "services": "svcs"}

def messy(text):
    words = text.lower().replace("?", "").replace(",", "").split()
    words = [ABBR.get(w, w) if rng.random() < 0.6 else w for w in words]
    idx = [i for i, w in enumerate(words) if len(w) >= 5 and w.isalpha()]
    for i in rng.sample(idx, k=min(len(idx), rng.choice([1, 2]))):
        w = words[i]; j = rng.randrange(1, len(w) - 2)
        words[i] = w[:j] + w[j + 1] + w[j] + w[j + 2:]
    out = " ".join(words)
    return (rng.choice(["", "", "pls ", "hey ", "quick q: ", "asap "]) + out +
            rng.choice(["", "", " thx", " asap", " plz"])).strip()

def norm(s):
    return re.sub(r"[^a-z0-9 ]", "", s.lower()).strip()

seen = {norm(p) for p in A["prompt"]}
ROTATION = ["shared_vocab", "no_cue", "near_miss", "vocab_shift", "messy",
            "shared_vocab", "no_cue", "near_miss", "messy", "cross_role"]
BASE_FOR_MESSY = ["shared_vocab", "no_cue", "vocab_shift"]

def make_one(tool, cat):
    """Return (prompt, family_id, category) or None if it could not make a new unique one."""
    for _ in range(40):
        if cat == "messy":
            base_cat = rng.choice(BASE_FOR_MESSY)
            pool = FAM[(base_cat, tool)]
            k = rng.randrange(len(pool))
            text = messy(fill(pool[k])); fam = f"{base_cat}:{tool}:{k}"
        else:
            pool = FAM.get((cat, tool))
            if not pool:
                return None
            k = rng.randrange(len(pool))
            text = fill(pool[k]); fam = f"{cat}:{tool}:{k}"
        if norm(text) not in seen:
            seen.add(norm(text))
            return text, fam, cat
    return None

new_rows = []
for tool in TOOLS:
    made, turn, stalls = 0, 0, 0
    while made < need[tool]:
        cat = ROTATION[turn % len(ROTATION)]; turn += 1
        got = make_one(tool, cat)
        if got is None:
            got = make_one(tool, "messy")      # fallback keeps the count on target
            if got is None:
                stalls += 1
                assert stalls < 200, f"cannot reach target for {tool}"
                continue
        text, fam, c = got
        new_rows.append({"prompt": text, "tool_ground_truth": tool,
                         "tool_predicted": None, "agent_role": AGENT_OF[tool],
                         "difficulty": "hard", "category": c,
                         "dataset": "B", "family": fam})
        made += 1

B_new = pd.DataFrame(new_rows)
print("new rows:", len(B_new))
print(B_new["category"].value_counts().to_dict())

new rows: 860
{'shared_vocab': 177, 'messy': 176, 'near_miss': 173, 'no_cue': 171, 'vocab_shift': 87, 'cross_role': 76}


## 4. Combine with dataset A

A rows are kept exactly as they were (prompts, labels and the stored Llama predictions).
They get a `family` id from their wording with numbers and filler stripped (approximate; it is
only used to group near-identical templates for held-out-template evaluation).

In [5]:
FILLERS = ["the target vm", "the target machine", "the defender host", "the target",
           "the defender", "the attacker ip", "the suspicious ip", "the malicious host",
           "the flagged address", "the web server", "this host", "this box", "this machine",
           "on the server", "on the defender", "on this host", "right now", "recently",
           "today", "since boot", "in the last hour", "this morning", "over the past day",
           "just now", "lately", "here"]

def a_family(p):
    p = p.lower()
    p = re.sub(r"\d+", " # ", p)
    for f in FILLERS:
        p = p.replace(f, " ")
    return "A:" + re.sub(r"[^a-z#]+", " ", p).strip()

A2 = A[["prompt", "tool_ground_truth", "tool_predicted", "agent_role", "difficulty",
        "category"]].copy()
A2["dataset"] = "A"
A2["family"] = A2["prompt"].map(a_family)

cols = ["prompt", "tool_ground_truth", "tool_predicted", "agent_role", "difficulty",
        "category", "dataset", "family"]
B = pd.concat([A2[cols], B_new[cols]], ignore_index=True)
print("dataset B:", len(B), "rows | A:", (B.dataset == "A").sum(), "| new:", (B.dataset == "B").sum())

dataset B: 1500 rows | A: 640 | new: 860


## 5. Automated checks

These replace manual review. They do not prove the labels are right; they catch duplicates,
miscounts, and wording that works against a label.

In [6]:
# 1. totals and A unchanged
assert len(B) == 1500
assert (B["dataset"] == "A").sum() == 640
assert B.loc[B.dataset == "A", "prompt"].tolist() == A["prompt"].tolist()
assert B.loc[B.dataset == "A", "tool_ground_truth"].tolist() == A["tool_ground_truth"].tolist()
print("OK: 1500 rows, dataset A unchanged")

# 2. exact duplicates
dups = B["prompt"].map(norm).duplicated().sum()
assert dups == 0, f"{dups} duplicate prompts"
print("OK: no duplicate prompts (after normalizing case and punctuation)")

# 3. every B template family maps to one tool
fam_tools = B_new.groupby("family")["tool_ground_truth"].nunique()
assert (fam_tools == 1).all()
print("OK: each new template family maps to exactly one tool")

# 4. counts
print("\nrows per tool (A + new = total):")
tab = pd.DataFrame({"A": B[B.dataset == "A"]["tool_ground_truth"].value_counts(),
                    "new": B[B.dataset == "B"]["tool_ground_truth"].value_counts()})
tab["total"] = tab["A"] + tab["new"]
print(tab.to_string())
print("\nby role:", B["agent_role"].value_counts().to_dict())
print("difficulty:", B["difficulty"].value_counts().to_dict())
print("category:", B["category"].value_counts().to_dict())

OK: 1500 rows, dataset A unchanged
OK: no duplicate prompts (after normalizing case and punctuation)
OK: each new template family maps to exactly one tool

rows per tool (A + new = total):
                     A  new  total
tool_ground_truth                 
BlockIP             55   81    136
CheckFailedLogins   49   88    137
CheckVulnerability  81   55    136
GetSystemInfo       49   88    137
ListProcesses       55   81    136
ListeningPorts      60   76    136
NmapScan            67   69    136
PortScan            68   68    136
ReadAuthLog         54   83    137
ReadSyslog          57   79    136
SSHConnect          45   92    137

by role: {'defender': 682, 'attacker': 545, 'shared': 273}
difficulty: {'hard': 1060, 'easy': 440}
category: {'direct': 440, 'shared_vocab': 177, 'messy': 176, 'near_miss': 173, 'no_cue': 171, 'vocab_shift': 87, 'cross_role': 76, 'opposite': 40, 'ambiguous': 40, 'multistep': 40, 'natural': 40, 'trick': 40}


In [7]:
# 5. near-duplicate report (token overlap), report only
toks = [set(norm(p).split()) for p in B["prompt"]]
pairs = []
for i in range(len(toks)):
    for j in range(i + 1, len(toks)):
        u = len(toks[i] | toks[j])
        if u and len(toks[i] & toks[j]) / u >= 0.9:
            pairs.append((i, j))
print(f"near-duplicate pairs (token overlap >= 0.9): {len(pairs)} of "
      f"{len(toks) * (len(toks) - 1) // 2} possible")
for i, j in pairs[:5]:
    print("  ", repr(B.prompt[i]), "|", repr(B.prompt[j]))

# 6. conflict report: cue words that belong to a different tool (report only)
CUES = {"NmapScan": ["nmap"], "CheckFailedLogins": ["failed", "failure", "brute"],
        "ReadSyslog": ["syslog"], "ReadAuthLog": ["auth"], "ListeningPorts": ["listening"],
        "ListProcesses": ["process"], "BlockIP": ["block"], "SSHConnect": [],
        "CheckVulnerability": ["cve", "exploit", "vulnerab"],
        "GetSystemInfo": ["uptime", "kernel", "hostname"], "PortScan": []}
rows = []
for _, r in B[B.dataset == "B"].iterrows():
    low = r.prompt.lower()
    for other, words in CUES.items():
        if other != r.tool_ground_truth and any(w in low for w in words):
            rows.append((r.tool_ground_truth, other))
conf = Counter(rows)
print(f"\nnew rows containing another tool's cue word: {sum(conf.values())}")
for (lab, other), n in conf.most_common(8):
    print(f"   labeled {lab:18} contains a {other} cue: {n}")
print("(expected for shared_vocab / near_miss rows, which are built that way on purpose)")

near-duplicate pairs (token overlap >= 0.9): 64 of 1124250 possible
   'Do a port scan of the defender that also reports the service version on each port' | 'Do a port scan of the defender host that also reports the service version on each port'
   'Do a service and version detection scan on the target VM' | 'Do a service and version detection scan on the target'
   'Run nmap on the target VM but I only care which ports are open, not versions' | 'Run nmap on the target but I only care which ports are open, not versions'
   'Use nmap -sV to detect software versions on the target machine' | 'Use nmap -sV to detect software versions on the target'
   "First scan the defender for open ports, then we'll look at versions" | "First scan the defender host for open ports, then we'll look at versions"

new rows containing another tool's cue word: 94
   labeled CheckFailedLogins  contains a ReadAuthLog cue: 32
   labeled ReadSyslog         contains a ReadAuthLog cue: 18
   labeled PortScan       

In [8]:
# 7. a few samples per family type for a quick read
for cat in ["shared_vocab", "no_cue", "near_miss", "vocab_shift", "cross_role", "messy"]:
    print(f"--- {cat} ---")
    sub = B_new[B_new.category == cat].sample(min(5, (B_new.category == cat).sum()),
                                              random_state=1)
    for _, r in sub.iterrows():
        print(f"   [{r.tool_ground_truth:18}] {r.prompt}")

--- shared_vocab ---
   [ListeningPorts    ] List what is bound to a port since boot
   [PortScan          ] Only tell me the port numbers that are open on that box
   [CheckFailedLogins ] From the auth log, just the failed logins
   [ListProcesses     ] Top CPU users on the server, not network ports
   [GetSystemInfo     ] Which kernel is this, and how long has it been up?
--- no_cue ---
   [NmapScan          ] What software does 192.168.56.102 expose today?
   [PortScan          ] What can I reach on the defender from here?
   [CheckFailedLogins ] Are there signs of brute force this morning?
   [ListProcesses     ] What is eating the CPU?
   [GetSystemInfo     ] Tell me about this system today
--- near_miss ---
   [ListeningPorts    ] What is bound to which port?
   [PortScan          ] nmap 10.0.0.114 and give me only open ports
   [CheckFailedLogins ] Which login attempts in auth.log failed here?
   [ListProcesses     ] Show everything that is running on this host
   [GetSystemInfo

## 6. Save

`tool_predicted` is filled for A (the stored Llama predictions) and empty for the new rows.
Notebook 05 fills it with Llama 3.2 predictions.

In [9]:
B.to_csv("cyber_dataset_B.csv", index=False)
print("saved cyber_dataset_B.csv:", B.shape)
B.head()

saved cyber_dataset_B.csv: (1500, 8)


,prompt,tool_ground_truth,tool_predicted,agent_role,difficulty,category,dataset,family
0,Do a port scan of the defender that also repor...,NmapScan,PortScan,attacker,hard,opposite,A,A:do a port scan of that also reports the serv...
1,Run nmap -p 1-1000 against 192.168.56.102,PortScan,NmapScan,attacker,easy,direct,A,A:run nmap p # # against # # # #
2,Scan ports 1-1000 on the defender host,PortScan,NmapScan,attacker,easy,direct,A,A:scan ports # # on
3,Is Apache 2.4.49 vulnerable to any known explo...,CheckVulnerability,CheckVulnerability,attacker,easy,direct,A,A:is apache # # # vulnerable to any known expl...
4,Is ssh 1.18 vulnerable to any known exploits?,CheckVulnerability,CheckVulnerability,attacker,easy,direct,A,A:is ssh # # vulnerable to any known exploits
